In [1]:
import os

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

from experiments import experiment_mlp_gn_coverage

## Sample vs IPOPT Lambda Selector at K=8 — Head-to-Head

This section directly compares the two outer-loop λ selectors on the same K=8 problem.
Both methods run the same inner-loop budget (`max_inner=25`) and the same initial point `x₀=0`;
the only difference is how each outer iteration chooses the next λ to target.

**Problem:** K=8, p=10, n=40, h=8, seed=10 (d=160).  
n is raised to 40 (≥ K×5) to ensure every class has enough samples for a non-zero Lipschitz estimate —
with K=8 and n=20, random label assignment can leave some classes empty, causing L[i]=0 and ZeroDivisionError.

**Methods compared:**

| | Sample | IPOPT |
|--|--|--|
| λ selection | 512 random Dirichlet draws + structured starts | multi-start IPOPT NLP, 20 starts |
| Cost per outer | cheap (vectorised sweep, ~0.01 s) | expensive (20 × interior-point solves) |
| `max_outer` | 500 | 200 |
| Expected runtime | < 5 min | ~20–40 min |

A coarse baseline (r=3, only 120 simplex grid points) is run first solely to produce the green
reference line; it is not the main comparison target.

**What to look for:**
- **Right panel (grad_evals):** same inner budget per outer step, so the gap between curves
  shows whether IPOPT's more precise λ\* translates to faster GN\* decay per gradient evaluation.
- **Left panel (CPU time):** wall-clock tradeoff — sample completes many more outer iterations
  in the same time, while IPOPT spends more per iteration finding a harder direction.

In [2]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from objectives import make_mlp_nonconvex
from baseline import uniform_discretisation
from algorithm import algorithm_adaptive

K = 8
p = 10
n = 40           # K*5 — avoids L[i]=0 when a class gets 0 samples
h = 8
seed = 10

coarse_resolution = 3   # C(10,7)=120 grid points — fast reference for K=8
n_passes = 15
steps_per_point_per_pass = 50
eval_every_n_grads = 2000
max_inner = 25           # default for K != 4

max_outer_sample = 500
lambda_random_starts = 512

max_outer_ipopt = 200
lambda_max_starts_ipopt = 20

# ── Build problem ──────────────────────────────────────────────────────────
d = h * p + h + K * h + K
objs, grads, L, joint = make_mlp_nonconvex(K=K, p=p, n=n, h=h, seed=seed)
x0 = np.zeros(d)
print(f"K={K}, p={p}, n={n}, h={h}, d={d}  |  L={np.round(L, 3)}")

# ── Baseline (coarse, for reference line only) ─────────────────────────────
print("\n[baseline — coarse reference r=3] ...")
bl = uniform_discretisation(
    K=K, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    resolution=coarse_resolution, n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    coverage_mode="gn", joint_oracle=joint, verbose=True)

# ── Adaptive: sample ───────────────────────────────────────────────────────
print("\n[adaptive — sample (512 draws)] ...")
a_sample = algorithm_adaptive(
    K=K, d=d, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    mode="gn",
    max_outer=max_outer_sample, max_inner=max_inner,
    eval_every_n_grads=eval_every_n_grads,
    target_cov=bl["cov_history"][-1],
    lambda_selector="sample",
    lambda_random_starts=lambda_random_starts,
    joint_oracle=joint, verbose=True)

# ── Adaptive: IPOPT ────────────────────────────────────────────────────────
print("\n[adaptive — IPOPT (20 starts)] ...")
a_ipopt = algorithm_adaptive(
    K=K, d=d, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    mode="gn",
    max_outer=max_outer_ipopt, max_inner=max_inner,
    eval_every_n_grads=eval_every_n_grads,
    target_cov=bl["cov_history"][-1],
    lambda_selector="optimize",
    lambda_max_starts=lambda_max_starts_ipopt,
    joint_oracle=joint, verbose=True)

# ── Summary ────────────────────────────────────────────────────────────────
final_err = bl["cov_history"][-1]
print(f"\n  BL     final GN* = {final_err:.4e}  "
      f"(ge={bl['grad_evals_history'][-1]}, cpu={bl['cpu_times'][-1]:.2f}s)")
print(f"  Sample final GN* = {a_sample['cov_history'][-1]:.4e}  "
      f"(ge={a_sample['grad_evals_history'][-1]}, cpu={a_sample['cpu_times'][-1]:.2f}s, "
      f"bundle={a_sample['bundle'].m})")
print(f"  IPOPT  final GN* = {a_ipopt['cov_history'][-1]:.4e}  "
      f"(ge={a_ipopt['grad_evals_history'][-1]}, cpu={a_ipopt['cpu_times'][-1]:.2f}s, "
      f"bundle={a_ipopt['bundle'].m})")

# ── Plot ───────────────────────────────────────────────────────────────────
ylabel = (r"$\sup_{\lambda\in\Delta_K} "
          r"[\min_{x_i\in\mathcal{B}_m} \|\nabla F_\lambda(x_i)\|^2]$")
fig, (ax_t, ax_g) = plt.subplots(1, 2, figsize=(12, 4.6))

for ax in (ax_t, ax_g):
    ax.axhline(final_err, color="#2ca02c", ls="--", lw=1.4,
               label=f"baseline final error = {final_err:.3e}")

ax_t.plot(a_sample["cpu_times"], a_sample["cov_history"],
          color="#1f77b4", marker="o", ms=4, lw=1.8,
          label=f"adaptive (sample, {lambda_random_starts} draws)")
ax_t.plot(a_ipopt["cpu_times"], a_ipopt["cov_history"],
          color="#ff7f0e", marker="^", ms=4, lw=1.8,
          label=f"adaptive (IPOPT, {lambda_max_starts_ipopt} starts)")

ax_g.plot(a_sample["grad_evals_history"], a_sample["cov_history"],
          color="#1f77b4", marker="o", ms=4, lw=1.8,
          label=f"adaptive (sample, {lambda_random_starts} draws)")
ax_g.plot(a_ipopt["grad_evals_history"], a_ipopt["cov_history"],
          color="#ff7f0e", marker="^", ms=4, lw=1.8,
          label=f"adaptive (IPOPT, {lambda_max_starts_ipopt} starts)")

ax_t.set_xlabel("CPU time (s)")
ax_g.set_xlabel("total gradient evaluations")
for ax in (ax_t, ax_g):
    ax.set_ylabel(ylabel)
    ax.set_yscale("log")
    ax.grid(True, which="both", alpha=0.25)
    ax.legend(frameon=False, fontsize=9)
ax_t.set_title("worst-case squared gradient norm vs CPU time")
ax_g.set_title("worst-case squared gradient norm vs gradient evals")

title = f"MLP K={K}, p={p}, n={n}, h={h}, d={d} — sample vs IPOPT"
fig.suptitle(title, fontsize=12)
fig.tight_layout(rect=(0, 0, 1, 0.96))

out_path = f"mlp_K{K}_p{p}_n{n}_h{h}_sample_vs_ipopt.png"
fig.savefig(out_path, dpi=150, bbox_inches="tight")
fig


K=8, p=10, n=40, h=8, d=160  |  L=[5.589 2.983 1.726 1.975 4.181 2.972 1.992 4.022]

[baseline — coarse reference r=3] ...
  Baseline pass 0/15 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=8.7500e-01
  Baseline pass 1/15 | t=1.37s | iters=6000 | grad_evals=48000 | worst-case pc=1.4935e-01
  Baseline pass 2/15 | t=2.73s | iters=12000 | grad_evals=96000 | worst-case pc=1.7199e-01
  Baseline pass 3/15 | t=4.09s | iters=18000 | grad_evals=144000 | worst-case pc=1.8412e-01
  Baseline pass 4/15 | t=5.39s | iters=24000 | grad_evals=192000 | worst-case pc=1.9022e-01
  Baseline pass 5/15 | t=6.74s | iters=30000 | grad_evals=240000 | worst-case pc=1.9385e-01
  Baseline pass 6/15 | t=8.04s | iters=36000 | grad_evals=288000 | worst-case pc=1.9623e-01
  Baseline pass 7/15 | t=9.38s | iters=42000 | grad_evals=336000 | worst-case pc=1.9793e-01
  Baseline pass 8/15 | t=10.77s | iters=48000 | grad_evals=384000 | worst-case pc=1.9920e-01
  Baseline pass 9/15 | t=12.13s | iters=54000 | grad_evals=4

<Figure size 1200x460 with 2 Axes>

## K=10 — Sample vs IPOPT，固定 grad_eval 预算，禁用早停

### 实验动机

K=8 实验中 sample 胜出的原因是目标太容易（baseline r=3 只有 120 个格点，GN* 参考线高达 0.20），
sample 仅用 ~50 次外迭代就触发了早停，IPOPT 的优势根本没来得及体现。

**IPOPT 的真正优势条件**：K 足够大，使随机 Dirichlet 采样无法覆盖单纯形顶点附近的区域。

| | K=8（已跑） | K=10（本节） |
|--|--|--|
| 单纯形维度 | 7D | 9D |
| P(lam_k > 0.8) under Dirichlet(1,...,1) | ~1.5e-5 | ~5e-7 |
| 512 个样本中平均命中近顶点次数 | ~0.008 | ~0.0003 |
| IPOPT 结构化起点覆盖全部顶点 | Yes | Yes |

在 K=10 的 9 维单纯形上，512 次随机采样基本不可能落在近顶点区域；
IPOPT 的结构化起点（顶点、近角点、边中点）每步都会精确搜索这些区域。

### 关键设计变化

- **禁用早停**（`target_cov=1e-10`）：两个方法跑完全相同的 `max_outer=200` 步，
  比较同等 grad_eval 预算下谁的 GN* 更低。
- **n=50**（K x 5）：保证每个类至少有足够样本，避免 L[i]=0 崩溃。
- **IPOPT `lambda_max_starts=25`**：K=10 全量结构化起点共 67 个，
  截到 25 以控制每步时间约 15-20 秒。

### 预期运行时间

| 阶段 | 估算 |
|------|------|
| Baseline (r=2, 55 格点) | ~1 分钟 |
| Sample (max_outer=200) | ~5 分钟 |
| IPOPT (max_outer=200, 25 starts) | ~15-20 分钟 |
| **合计** | **~25 分钟** |

### 预期结论

- **右图（grad_evals）**：IPOPT 曲线应低于 sample 曲线——相同梯度计算量下，
  IPOPT 找到更难的 lambda*，GN* 下降更快。
- **左图（CPU time）**：sample 仍会更快，每步开销远低于 IPOPT。
  左图体现速度权衡，右图才是算法质量的公平对比。

In [3]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from objectives import make_mlp_nonconvex
from baseline import uniform_discretisation
from algorithm import algorithm_adaptive

K = 10
p = 10
n = 50           # K*5 — 保证每类有足够样本，避免 L[i]=0
h = 8
seed = 10

coarse_resolution = 2    # C(11,2)=55 格点，K=10 时 baseline 快速完成
n_passes = 10
steps_per_point_per_pass = 50
eval_every_n_grads = 2000
max_inner = 25

max_outer = 200          # 两个方法完全一样 -> 同等 grad_eval 预算
NO_EARLY_STOP = 1e-10    # 极小值，实际上禁止早停

lambda_random_starts = 512
lambda_max_starts_ipopt = 25   # 全量 67 个起点截到 25，控制每步时间

# Build problem
d = h * p + h + K * h + K
objs, grads, L, joint = make_mlp_nonconvex(K=K, p=p, n=n, h=h, seed=seed)
x0 = np.zeros(d)
print(f'K={K}, p={p}, n={n}, h={h}, d={d}  |  L={np.round(L, 3)}')

# Baseline (coarse, for reference line only)
print('\n[baseline — r=2, 55 grid points] ...')
bl = uniform_discretisation(
    K=K, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    resolution=coarse_resolution, n_passes=n_passes,
    steps_per_point_per_pass=steps_per_point_per_pass,
    eval_every_n_grads=eval_every_n_grads,
    coverage_mode='gn', joint_oracle=joint, verbose=True)

# Adaptive: sample
print('\n[adaptive — sample (512 draws), no early stop] ...')
a_sample = algorithm_adaptive(
    K=K, d=d, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    mode='gn',
    max_outer=max_outer, max_inner=max_inner,
    eval_every_n_grads=eval_every_n_grads,
    target_cov=NO_EARLY_STOP,
    lambda_selector='sample',
    lambda_random_starts=lambda_random_starts,
    joint_oracle=joint, verbose=True)

# Adaptive: IPOPT
print('\n[adaptive — IPOPT (25 starts), no early stop] ...')
a_ipopt = algorithm_adaptive(
    K=K, d=d, objectives=objs, grad_objectives=grads, L=L, x0=x0,
    mode='gn',
    max_outer=max_outer, max_inner=max_inner,
    eval_every_n_grads=eval_every_n_grads,
    target_cov=NO_EARLY_STOP,
    lambda_selector='optimize',
    lambda_max_starts=lambda_max_starts_ipopt,
    joint_oracle=joint, verbose=True)

# Summary
final_err = bl['cov_history'][-1]
print(f'\n  BL     final GN* = {final_err:.4e}  '
      f'(ge={bl["grad_evals_history"][-1]}, cpu={bl["cpu_times"][-1]:.2f}s)')
print(f'  Sample final GN* = {a_sample["cov_history"][-1]:.4e}  '
      f'(ge={a_sample["grad_evals_history"][-1]}, cpu={a_sample["cpu_times"][-1]:.2f}s, '
      f'bundle={a_sample["bundle"].m})')
print(f'  IPOPT  final GN* = {a_ipopt["cov_history"][-1]:.4e}  '
      f'(ge={a_ipopt["grad_evals_history"][-1]}, cpu={a_ipopt["cpu_times"][-1]:.2f}s, '
      f'bundle={a_ipopt["bundle"].m})')

# Plot
ylabel = (r'$\sup_{\lambda\in\Delta_K} '
          r'[\min_{x_i\in\mathcal{B}_m} \|\nabla F_\lambda(x_i)\|^2]$')
fig, (ax_t, ax_g) = plt.subplots(1, 2, figsize=(12, 4.6))

for ax in (ax_t, ax_g):
    ax.axhline(final_err, color='#2ca02c', ls='--', lw=1.4,
               label=f'baseline final error = {final_err:.3e}')

ax_t.plot(a_sample['cpu_times'], a_sample['cov_history'],
          color='#1f77b4', marker='o', ms=4, lw=1.8,
          label=f'adaptive (sample, {lambda_random_starts} draws)')
ax_t.plot(a_ipopt['cpu_times'], a_ipopt['cov_history'],
          color='#ff7f0e', marker='^', ms=4, lw=1.8,
          label=f'adaptive (IPOPT, {lambda_max_starts_ipopt} starts)')

ax_g.plot(a_sample['grad_evals_history'], a_sample['cov_history'],
          color='#1f77b4', marker='o', ms=4, lw=1.8,
          label=f'adaptive (sample, {lambda_random_starts} draws)')
ax_g.plot(a_ipopt['grad_evals_history'], a_ipopt['cov_history'],
          color='#ff7f0e', marker='^', ms=4, lw=1.8,
          label=f'adaptive (IPOPT, {lambda_max_starts_ipopt} starts)')

ax_t.set_xlabel('CPU time (s)')
ax_g.set_xlabel('total gradient evaluations')
for ax in (ax_t, ax_g):
    ax.set_ylabel(ylabel)
    ax.set_yscale('log')
    ax.grid(True, which='both', alpha=0.25)
    ax.legend(frameon=False, fontsize=9)
ax_t.set_title('GN* vs CPU time  [speed comparison]')
ax_g.set_title('GN* vs gradient evals  [algorithm quality comparison]')

title = (f'MLP K={K}, p={p}, n={n}, h={h}, d={d} '
         f'— sample vs IPOPT (no early stop, max_outer={max_outer})')
fig.suptitle(title, fontsize=11)
fig.tight_layout(rect=(0, 0, 1, 0.96))

out_path = f'mlp_K{K}_p{p}_n{n}_h{h}_sample_vs_ipopt_fixed_budget.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
fig


K=10, p=10, n=50, h=8, d=178  |  L=[3.871 3.244 1.7   7.086 5.583 3.277 2.557 2.432 1.598 1.923]

[baseline — r=2, 55 grid points] ...
  Baseline pass 0/10 | t=0.00s | iters=0 | grad_evals=0 | worst-case pc=9.0000e-01
  Baseline pass 1/10 | t=0.79s | iters=2750 | grad_evals=27500 | worst-case pc=2.9177e-01
  Baseline pass 2/10 | t=1.58s | iters=5500 | grad_evals=55000 | worst-case pc=3.4656e-01
  Baseline pass 3/10 | t=2.35s | iters=8250 | grad_evals=82500 | worst-case pc=3.6496e-01
  Baseline pass 4/10 | t=3.11s | iters=11000 | grad_evals=110000 | worst-case pc=3.7399e-01
  Baseline pass 5/10 | t=3.86s | iters=13750 | grad_evals=137500 | worst-case pc=3.7932e-01
  Baseline pass 6/10 | t=4.62s | iters=16500 | grad_evals=165000 | worst-case pc=3.8283e-01
  Baseline pass 7/10 | t=5.40s | iters=19250 | grad_evals=192500 | worst-case pc=3.8532e-01
  Baseline pass 8/10 | t=6.16s | iters=22000 | grad_evals=220000 | worst-case pc=3.8717e-01
  Baseline pass 9/10 | t=6.93s | iters=24750 | grad_

<Figure size 1200x460 with 2 Axes>